## Surface Code decoders comparison

In [1]:
import stim
import numpy as np
from beliefmatching import BeliefMatching
from adaptiveQRL.decompose_errors import decompose_errors_for_stim_surface_code_coords

0. Construct the surface code Stim circui and the corresponding error model

In [4]:
d = 5

'''
num_shots = 240_000_000
chunk_shots = 24_000_000        # per-chunk batch size; tune to fit in RAM
p = 0.0002
'''

p = 0.001
num_shots = 2_000_000
chunk_shots = 200_000

circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=d,
    distance=d,
    before_round_data_depolarization=p,
    before_measure_flip_probability=p,
    after_reset_flip_probability=p,
    after_clifford_depolarization=p,
)

sampler = circuit.compile_detector_sampler()
dem = decompose_errors_for_stim_surface_code_coords(
    circuit.detector_error_model(decompose_errors=False)
)

1. Evaluate Belief Matching with different number of belief propagation iterations

In [ ]:
bm = BeliefMatching(dem, max_bp_iters=10)

# Chunked sampling + decoding so we don't allocate (num_shots, n_detectors) up front.
num_errors = 0
shots_done = 0
while shots_done < num_shots:
    this_chunk = min(chunk_shots, num_shots - shots_done)
    shots, observables = sampler.sample(this_chunk, separate_observables=True)
    predicted = bm.decode_batch(shots)
    num_errors += int(np.sum(np.any(predicted != observables, axis=1)))
    shots_done += this_chunk
    print(f"  {shots_done:,}/{num_shots:,} shots done  errors={num_errors}", flush=True)

ler = num_errors / num_shots
std = np.sqrt(ler * (1.0 - ler) / num_shots)  # binomial SE, same as the other scripts
print(f"\nLER = {num_errors}/{num_shots} = {ler:.8e} +/- {std:.2e}")

In [3]:
import pymatching

# Standard MWPM on the same (decomposed) DEM, no correlation post-processing.
# `Matching.from_detector_error_model` builds a matching that knows about the
# observables, so `decode_batch` returns predicted observable flips directly
# (shape (n_shots, n_observables)) -- no edge -> observable conversion needed.
m_mwpm = pymatching.Matching.from_detector_error_model(dem)

num_errors_mwpm = 0
shots_done = 0
while shots_done < num_shots:
    this_chunk = min(chunk_shots, num_shots - shots_done)
    shots, observables = sampler.sample(this_chunk, separate_observables=True)
    pred_obs = np.asarray(
        m_mwpm.decode_batch(shots, enable_correlations=False)
    )                                                                 # shape (this_chunk, n_obs)
    num_errors_mwpm += int(np.sum(np.any(pred_obs != observables, axis=1)))
    shots_done += this_chunk
    print(f"  MWPM  {shots_done:,}/{num_shots:,} shots  errors={num_errors_mwpm}", flush=True)

ler_mwpm = num_errors_mwpm / num_shots
std_mwpm = np.sqrt(ler_mwpm * (1.0 - ler_mwpm) / num_shots)
print(f"\nStandard MWPM LER = {num_errors_mwpm}/{num_shots} = {ler_mwpm:.8e} +/- {std_mwpm:.2e}")

  MWPM  24,000,000/240,000,000 shots  errors=24
  MWPM  48,000,000/240,000,000 shots  errors=51
  MWPM  72,000,000/240,000,000 shots  errors=74
  MWPM  96,000,000/240,000,000 shots  errors=113
  MWPM  120,000,000/240,000,000 shots  errors=138
  MWPM  144,000,000/240,000,000 shots  errors=158
  MWPM  168,000,000/240,000,000 shots  errors=178
  MWPM  192,000,000/240,000,000 shots  errors=214
  MWPM  216,000,000/240,000,000 shots  errors=241
  MWPM  240,000,000/240,000,000 shots  errors=261

Standard MWPM LER = 261/240000000 = 1.08750000e-06 +/- 6.73e-08


In [5]:
import pymatching

# Correlated Matching via pymatching's enable_correlations=True path on the same
# decomposed DEM. The fork returns observable predictions directly in this mode.
m_corr = pymatching.Matching.from_detector_error_model(dem, enable_correlations=True)

num_errors_corr = 0
shots_done = 0
while shots_done < num_shots:
    this_chunk = min(chunk_shots, num_shots - shots_done)
    shots, observables = sampler.sample(this_chunk, separate_observables=True)
    pred_obs = np.asarray(
        m_corr.decode_batch(shots, enable_correlations=True)
    )                                                                 # shape (this_chunk, n_obs)
    num_errors_corr += int(np.sum(np.any(pred_obs != observables, axis=1)))
    shots_done += this_chunk
    print(f"  Corr  {shots_done:,}/{num_shots:,} shots  errors={num_errors_corr}", flush=True)

ler_corr = num_errors_corr / num_shots
std_corr = np.sqrt(ler_corr * (1.0 - ler_corr) / num_shots)
print(f"\nCorrelated Matching LER = {num_errors_corr}/{num_shots} = {ler_corr:.8e} +/- {std_corr:.2e}")

  Corr  24,000,000/240,000,000 shots  errors=25
  Corr  48,000,000/240,000,000 shots  errors=55
  Corr  72,000,000/240,000,000 shots  errors=85
  Corr  96,000,000/240,000,000 shots  errors=106
  Corr  120,000,000/240,000,000 shots  errors=125
  Corr  144,000,000/240,000,000 shots  errors=153
  Corr  168,000,000/240,000,000 shots  errors=169
  Corr  192,000,000/240,000,000 shots  errors=198
  Corr  216,000,000/240,000,000 shots  errors=223
  Corr  240,000,000/240,000,000 shots  errors=242

Correlated Matching LER = 242/240000000 = 1.00833333e-06 +/- 6.48e-08


4. Evaluate Neural Correlated Matching

In [5]:
from adaptiveQRL.neural_correlated_matching import NeuralCorrelatedMatching

# Trained SAC-GNN policy on top of two-pass MWPM. Constructor args (hidden_dim,
# n_layers, action_scale, local_action_hops, use_endpoint_firing) MUST match
# what the model was trained with. The model path is relative to the notebook
# (not the repo root), so go up one level.
ncm = NeuralCorrelatedMatching(
    dem,
    model_path="../models/qec_graph_optuna_run_d5_trial_0000_best.pth",
    hidden_dim=256,
    n_layers=1,
    alpha=0.01,
    action_scale=5.0,
    bypass_threshold=2,
    local_action_hops=1,
    use_endpoint_firing=False,
)

num_errors_ncm = 0
shots_done = 0
while shots_done < num_shots:
    this_chunk = min(chunk_shots, num_shots - shots_done)
    shots, observables = sampler.sample(this_chunk, separate_observables=True)
    pred_obs = ncm.decode_batch(shots)
    num_errors_ncm += int(np.sum(np.any(pred_obs != observables, axis=1)))
    shots_done += this_chunk
    print(f"  NCM   {shots_done:,}/{num_shots:,} shots  errors={num_errors_ncm}", flush=True)

ler_ncm = num_errors_ncm / num_shots
std_ncm = np.sqrt(ler_ncm * (1.0 - ler_ncm) / num_shots)
print(f"\nNeural Correlated Matching LER = {num_errors_ncm}/{num_shots} = {ler_ncm:.8e} +/- {std_ncm:.2e}")

[*] Models successfully loaded from ../models/qec_graph_optuna_run_d5_trial_0000_best.pth
  NCM   200,000/2,000,000 shots  errors=17
  NCM   400,000/2,000,000 shots  errors=41
  NCM   600,000/2,000,000 shots  errors=66
  NCM   800,000/2,000,000 shots  errors=81
  NCM   1,000,000/2,000,000 shots  errors=105
  NCM   1,200,000/2,000,000 shots  errors=126
  NCM   1,400,000/2,000,000 shots  errors=140
  NCM   1,600,000/2,000,000 shots  errors=157
  NCM   1,800,000/2,000,000 shots  errors=184
  NCM   2,000,000/2,000,000 shots  errors=199

Neural Correlated Matching LER = 199/2000000 = 9.95000000e-05 +/- 7.05e-06
